# Build 04-03 · Does mitigation shrink the SFP loop? (within-version + cross-version ShapDiDDelta)

## What this measures, in order

1. **Within-version (Step 1)** — for v2 alone and v3 alone: does swapping the training labels
   (baseline -> a corrected axis) change the region(A/B) treatment-effect on SHAP concentration,
   on that version's OWN population? No era, no cross-version pairing — anchored at baseline
   (vs every axis) and at naive (vs every OTHER axis, isolating the label-change effect from the
   population-change effect `baseline->naive` already shows). Absorbs
   `shap_did/04_05_shap_did_within_version.ipynb` (retired 2026-09-15, folded in here).
2. **Cross-version (Step 2)** — does the region(A/B) gap grow MORE from v2 to v3 under one
   correction method than under another? Matching-tag only (v2's naive vs v3's naive, etc.) —
   never an arbitrary v2-tag vs v3-tag pair, which would compare two different correction
   methods to each other with no clean question behind it.
3. **Local RDD (Step 3)** — the Step 2 comparison again, restricted to a boundary band around τ,
   as a robustness check against region A/B having different covariate distributions.

**Region × tag (Step 1) and region × version (Step 2) are each a single DiD.** The
`ShapDiDDelta` reported for a PAIR of tags (`baseline->naive`, `naive->transport`, ...) is where
the real double-difference sits — Step 1's is (region × tag), Step 2's is (region × version ×
tag). Earlier versions of this notebook additionally split by `era` (early/late within one
version's own time window) before any of this — that was `04_02`'s `within_version_confound_
check` mechanism (a PARALLEL-TRENDS diagnostic, not an estimate), repurposed here by mistake as
if it were the headline measurement. Removed 2026-09-15: region × tag/version is the correct,
`04_02`-consistent design; `era`/`within_version_confound_check`-style checks stay in `04_02`,
not here.

In [ ]:
# §0 — setup (analysis .venv kernel)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import schema
import figstyle
import feature_alias
from estimator import concentration

figstyle.apply()
figstyle.FIG_DIR = ROOT / "figures" / "shap_did" / "04_03"
figstyle.ALIAS_SPLIT = True  # real-name/alias per-feature outputs -> real_named/ / alias/
pd.set_option("display.width", 160)
print("ROOT =", ROOT)
print("FIG_DIR =", figstyle.FIG_DIR)

### 실행 전 준비 — 이 노트북이 필요로 하는 `mitigated_attributions` 파일 목록

아래 파일들이 `src/data/real/mitigation/shap/<v>/`에 이미 있어야 이 노트북이 끝까지 돌아갑니다.
없으면 `00_SHAP.ipynb`를 그 버전 커널(`fttl-v2`/`fttl-v3`)에서, 아래 `MODEL_PATH`로 **SPLIT을
바꿔가며 두 번씩**(train 한 번, OOT 한 번) 돌리세요 — `00_SHAP.ipynb`의 "실행 전 설정" 셀 참고.
`OUT_PATH`는 항상 `None`(자동).

**v2 (커널 `fttl-v2`, SPLIT: `"train"` 그대로 / `"test"`로 바꿔서 각각 한 번씩)**
```
src/models/real/mitigated/v2_train_naive.pkl
src/models/real/mitigated/v2_train_transport.pkl
```

**v3 (커널 `fttl-v3`, SPLIT: `"train"` 그대로 / `"oot"`로 바꿔서 각각 한 번씩)**
```
src/models/real/mitigated/v3_train_naive.pkl
src/models/real/mitigated/v3_train_rarity_regime.pkl
src/models/real/mitigated/v3_train_rarity_fixed.pkl
src/models/real/mitigated/v3_train_transport.pkl
src/models/real/mitigated/v3_train_pnu_regime.pkl
src/models/real/mitigated/v3_train_pnu_fixed.pkl
```

총 16번 실행 (2×2 + 6×2). **baseline**(`MODEL_PATH=None`) SHAP도 train/OOT 둘 다 이미 있는지
확인하세요 — 없으면 같은 방식으로(단 `MODEL_PATH=None`) 한 번씩 돌려야 합니다. 어느 축이 아직
없으면 아래 코드가 그 축만 건너뛰고 나머지로 계속 진행합니다 (에러 아님, 안내 문구만 출력).

In [ ]:
# §1 — shared machinery. "baseline" is the unmitigated model (kind "attributions"); every other
# tag is one 03_02 correction axis (kind "mitigated_attributions" + that axis suffix -- the kind's
# own path carries no axis, see DATA_MODEL.md §5).
ID_COL = schema.CLAIM_ID
TAG_COLS = [schema.DATE, "score", schema.DECISION, "region", "era", schema.OBSERVED]

AXES = {"v2": ("naive", "transport"),
        "v3": ("naive", "rarity_regime", "rarity_fixed", "transport", "pnu_regime", "pnu_fixed")}
OOT = config.OOT_SPLIT          # {"v1": "val2", "v2": "test", "v3": "oot"}
TRAIN_SPLIT = {"v2": "train", "v3": "train"}


def attributions_path(version: str, split: str, tag: str) -> Path:
    if tag == "baseline":
        return config.path("attributions", version, split=split)
    base = config.path("mitigated_attributions", version, split=split)
    return base.with_name(f"{base.stem}_{tag}{base.suffix}")


def tag_available(version: str, split: str, tag: str) -> bool:
    return attributions_path(version, split, tag).exists()


def available_tags(version: str, split: str) -> list[str]:
    """"baseline" (asserted to exist) + whichever of that version's AXES have SHAP on disk."""
    base_p = attributions_path(version, split, "baseline")
    assert base_p.exists(), f"[{version}/{split}] no baseline attributions at {base_p} -- run 00_SHAP.ipynb first"
    found = ["baseline"] + [a for a in AXES[version] if tag_available(version, split, a)]
    missing = [a for a in AXES[version] if a not in found]
    if missing:
        print(f"[{version}/{split}] not yet computed: {missing} -- comparisons below use only {found[1:]}")
    return found


def load_region_table(version: str, split: str, tag: str) -> pd.DataFrame:
    """region(A/B) tags (04_01's shap_did_input) joined to `tag`'s attributions for the SAME
    claims. `era` rides along unused (dropped later by TAG_COLS) -- region is the only split this
    notebook uses now."""
    tags_df = pd.read_parquet(config.split_path("shap_did_input", version, split))
    attrs = pd.read_parquet(attributions_path(version, split, tag))
    return tags_df.merge(attrs, on=ID_COL, how="inner")


def restrict_to_corrector_targets(table: pd.DataFrame, version: str, split: str) -> pd.DataFrame:
    """Restrict to claims corrector_targets covers -- the population every corrected model for
    this (version, split) actually trained on (03_02 fits every RUN on the SAME corrector_targets
    file). Only meaningful for TRAIN: baseline was fit on the FULL population but corrected models
    were fit on this filtered subset, so comparing baseline's SHAP on the full population against
    a corrected model's SHAP on the same full population is partly out-of-fit-sample for the
    corrected side. OOT needs no such restriction -- no model, baseline or corrected, ever
    trained on OOT rows, so every tag is equally out-of-sample there already."""
    ctp = config.split_path("corrector_targets", version, split)
    if not ctp.is_file():
        print(f"[{version}/{split}] no corrector_targets -- skipping population restriction")
        return table
    covered = pd.read_parquet(ctp, columns=[ID_COL])[ID_COL]
    return table[table[ID_COL].isin(covered)]


def cell_simpson(table: pd.DataFrame, **filters) -> float:
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS if c in sub.columns]
    mabs = concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)
    return concentration.simpson(mabs)


def _tau_by_date(table: pd.DataFrame, version: str) -> pd.Series:
    dates = pd.to_datetime(table[schema.DATE])
    tau_by_date = {d: config.threshold_on(version, d) for d in dates.unique()}
    return dates.map(tau_by_date)


def local_band(table: pd.DataFrame, version: str, h: float) -> pd.DataFrame:
    """Band-restricted copy of `table`, with "region" OVERWRITTEN to the near-tau A'/B' tag
    (|score - tau| <= h, strict '>' above tau is B) -- reuses cell_simpson() unchanged, no
    separate region_local machinery needed."""
    dist = table["score"] - _tau_by_date(table, version)
    band = table[dist.abs() <= h].copy()
    band["region"] = np.where(dist[dist.abs() <= h] > 0, "B", "A")
    return band


def region_effect(version: str, split: str, tag: str, restrict_population: bool = False,
                  h: float | None = None) -> dict:
    """simpson(B) - simpson(A): the region-based "treatment effect" on SHAP concentration, for
    ONE (version, split, tag). h != None restricts to the boundary band first (Step 3)."""
    table = load_region_table(version, split, tag)
    if restrict_population:
        table = restrict_to_corrector_targets(table, version, split)
    if h is not None:
        table = local_band(table, version, h)
    sA, sB = cell_simpson(table, region="A"), cell_simpson(table, region="B")
    return {"n_claims": len(table), "simpson_A": sA, "simpson_B": sB, "region_effect": sB - sA}


def anchored_pairs(effects: dict) -> list[tuple[str, str]]:
    """baseline vs every OTHER computed tag, PLUS naive vs every tag besides baseline/naive --
    never every arbitrary pair (see the notebook intro: e.g. transport-vs-rarity_regime compares
    two different correction methods with no clean question behind it)."""
    tags = list(effects)
    pairs = [("baseline", t) for t in tags if t != "baseline"]
    if "naive" in tags:
        pairs += [("naive", t) for t in tags if t not in ("baseline", "naive")]
    return pairs


def shap_did_table(effects: dict, value_key: str = "region_effect") -> pd.DataFrame:
    """ShapDiDDelta for every anchored_pairs() pair, reading `value_key` out of each tag's dict
    in `effects` (region_effect() for Step 1, or a bare {tag: float} for Step 2/3)."""
    rows = []
    for a, b in anchored_pairs(effects):
        va = effects[a][value_key] if isinstance(effects[a], dict) else effects[a]
        vb = effects[b][value_key] if isinstance(effects[b], dict) else effects[b]
        delta = va - vb
        pct = delta / va if va else float("nan")
        rows.append({"pair": f"{a}->{b}", "a": va, "b": vb, "ShapDiDDelta": delta, "pct_erased": pct})
        print(f"ShapDiDDelta({a}->{b}) = {va:+.4f} - {vb:+.4f} = {delta:+.4f}"
              + (f"  (~{pct:.0%} erased)" if pd.notna(pct) else ""))
    return pd.DataFrame(rows).set_index("pair")

## Step 1 — within-version, anchored (no era, no cross-version pairing)

For each version alone, on its own population: `region_effect(tag) = simpson(B, tag) -
simpson(A, tag)`. `ShapDiDDelta(tag_a->tag_b) = region_effect(tag_a) - region_effect(tag_b)` --
a single DiD (region × tag). Run twice per version: **train** (population restricted to
`corrector_targets` -- the claims every corrected model for that version actually trained on) and
**OOT** (no restriction needed -- see `restrict_to_corrector_targets`'s docstring above).

In [ ]:
# §2 — Step 1: within-version anchored ShapDiDDelta, for (version, split) in
# {(v2,train), (v2,oot), (v3,train), (v3,oot)}
STEP1_POPULATIONS = [
    ("v2", TRAIN_SPLIT["v2"], True),
    ("v2", OOT["v2"], False),
    ("v3", TRAIN_SPLIT["v3"], True),
    ("v3", OOT["v3"], False),
]

step1_effects = {}
step1_tables = {}
for version, split, restrict in STEP1_POPULATIONS:
    print(f"\n=== {version} / {split} (population restricted: {restrict}) ===")
    tags = available_tags(version, split)
    effects = {t: region_effect(version, split, t, restrict_population=restrict) for t in tags}
    for t, e in effects.items():
        print(f"  [{t}] n={e['n_claims']:,}  simpson(A)={e['simpson_A']:.4f}  "
              f"simpson(B)={e['simpson_B']:.4f}  region_effect={e['region_effect']:+.4f}")
    table = shap_did_table(effects)
    step1_effects[(version, split)] = effects
    step1_tables[(version, split)] = table
    display(table.round(4))
    figstyle.save_table(table, f"{version}_{split}_04_03_step1_within_version")

## Step 2 — cross-version, matching tag only (v2 vs v3, on OOT)

`corruption_footprint[tag] = region_effect(v3, tag) - region_effect(v2, matched_tag)`, where
`matched_tag = tag` if v2 has that axis, else v2's `baseline` (v2 has only `{naive, transport}`
-- there is no v2 `rarity_regime` etc. to use, so those tags fall back with a printed warning,
same as before). `ShapDiDDelta(tag_a->tag_b) = corruption_footprint[tag_a] -
corruption_footprint[tag_b]` -- THIS is the real double-difference (region × version × tag);
`corruption_footprint` on its own is a single DiD (region × version), matching `04_02`'s
`cross_version_estimate` structure exactly, just repeated per tag instead of only for baseline.

**OOT only, never train, and this is a hard requirement, not a preference:**
- **Consistency**: `04_02`'s `cross_version_estimate` -- the "how much did the loop intensify
  from v2 to v3" question this Step answers per-tag -- is already defined on OOT. A train-based
  version here would not be the same estimand as `04_02`'s headline, just something that looks
  similar.
- **No population-matching machinery needed**: on train, baseline was fit on the FULL population
  while every corrected axis was fit on the `corrector_targets`-filtered subset (Step 1's
  `restrict_population` exists to fix exactly that asymmetry). On OOT, NO model -- baseline or
  corrected -- ever trained on any OOT row, so every tag is equally out-of-sample there. Cross-
  version comparison is naturally apples-to-apples on OOT; on train it would only be apples-to-
  apples with Step 1's extra restriction step, which is unnecessary complexity for a comparison
  that already has an established OOT convention to follow.

In [ ]:
# §3 — Step 2: cross-version, matching tag, OOT only
v3_tags = available_tags("v3", OOT["v3"])
corruption_footprint = {}
cf_detail = {}
for tag in v3_tags:
    v2_ok = tag == "baseline" or tag_available("v2", OOT["v2"], tag)
    v2_tag = tag if v2_ok else "baseline"
    if tag != "baseline":
        print(f"[{tag}] v2 {'FOUND' if v2_ok else 'NOT FOUND -> using v2 BASELINE'}")
    e_v2 = region_effect("v2", OOT["v2"], v2_tag)
    e_v3 = region_effect("v3", OOT["v3"], tag)
    cf = e_v3["region_effect"] - e_v2["region_effect"]
    corruption_footprint[tag] = cf
    cf_detail[tag] = {"v2_tag_used": v2_tag, "v2_region_effect": e_v2["region_effect"],
                      "v3_region_effect": e_v3["region_effect"], "corruption_footprint": cf}
    print(f"  corruption_footprint[{tag}] = {cf:+.4f}")

cf_table = pd.DataFrame(cf_detail).T
display(cf_table.round(4))
figstyle.save_table(cf_table, "04_03_step2_corruption_footprint")

print()
step2_pairs = shap_did_table(corruption_footprint)
display(step2_pairs.round(4))
figstyle.save_table(step2_pairs, "04_03_step2_shap_did_delta")

## Step 3 — local RDD (boundary-band) robustness check on Step 2

Same matching-tag design as Step 2, restricted to a grid-selected band `|score - tau| <= h`
around each row's own τ (`h` selected on the BASELINE OOT tables only, on cell counts alone --
never re-selected on a corrected table, so every tag's local check uses the identical partition).
Region A and B do not share a covariate distribution, so part of Step 2's whole-population
`corruption_footprint`/`ShapDiDDelta` could reflect that distributional gap rather than the
forced-label mechanism the corrector targets -- this section checks whether each Step 2 number
survives once the two sides are forced to look almost identical except for which side of τ they
scored on.

In [ ]:
# §4 — Step 3: select h, then repeat Step 2 exactly but with local_band() applied first
LOCAL_H_GRID = [0.005, 0.0075, 0.01, 0.015, 0.02, 0.03, 0.05, 0.075, 0.1]
LOCAL_MIN_CELL_N = 100
LOCAL_H_MAX = 0.05
LOCAL_H_FALLBACK = 0.01


def select_local_h() -> tuple[float, pd.DataFrame]:
    base = {"v2": load_region_table("v2", OOT["v2"], "baseline"),
            "v3": load_region_table("v3", OOT["v3"], "baseline")}
    dist = {v: t["score"] - _tau_by_date(t, v) for v, t in base.items()}
    rows = []
    for h in sorted(LOCAL_H_GRID):
        counts = {}
        for v, t in base.items():
            in_band = dist[v].abs() <= h
            region = np.where(dist[v][in_band] > 0, "B", "A")
            for r in ("A", "B"):
                counts[f"{v}_{r}"] = int((region == r).sum())
        rows.append({"h": h, **counts, "all_pass": min(counts.values()) >= LOCAL_MIN_CELL_N})
    bt = pd.DataFrame(rows).set_index("h")
    ok = bt.index[bt["all_pass"] & (bt.index <= LOCAL_H_MAX)]
    if len(ok):
        chosen = float(ok[0])
        print(f"selected h={chosen} (smallest with every v2/v3 x region cell >= {LOCAL_MIN_CELL_N}, "
              f"h<={LOCAL_H_MAX})")
    else:
        chosen = LOCAL_H_FALLBACK
        print(f"!! no h <= {LOCAL_H_MAX} reaches {LOCAL_MIN_CELL_N} claims per cell -- keeping "
              f"fallback h={LOCAL_H_FALLBACK}; read Step 3 as LOW-POWER.")
    display(bt)
    return chosen, bt


LOCAL_H, local_h_table = select_local_h()
figstyle.save_table(local_h_table, "04_03_step3_local_h_selection")

corruption_footprint_local = {}
for tag in v3_tags:
    v2_tag = cf_detail[tag]["v2_tag_used"]
    e_v2 = region_effect("v2", OOT["v2"], v2_tag, h=LOCAL_H)
    e_v3 = region_effect("v3", OOT["v3"], tag, h=LOCAL_H)
    cf_local = e_v3["region_effect"] - e_v2["region_effect"]
    corruption_footprint_local[tag] = cf_local
    print(f"corruption_footprint_local[{tag}] (h={LOCAL_H}) = {cf_local:+.4f}")

print()
step3_pairs = shap_did_table(corruption_footprint_local)
step3_pairs = step3_pairs.rename(columns={"a": "a_local", "b": "b_local",
                                          "ShapDiDDelta": "ShapDiDDelta_local",
                                          "pct_erased": "pct_erased_local"})
display(step3_pairs.round(4))
figstyle.save_table(step3_pairs, "04_03_step3_shap_did_delta_local")

whole_vs_local = step2_pairs[["ShapDiDDelta"]].join(step3_pairs[["ShapDiDDelta_local"]])
whole_vs_local["same_sign"] = (np.sign(whole_vs_local["ShapDiDDelta"])
                                == np.sign(whole_vs_local["ShapDiDDelta_local"]))
display(whole_vs_local.round(4))
figstyle.save_table(whole_vs_local, "04_03_whole_vs_local")
for pair, row in whole_vs_local.iterrows():
    verdict = ("survives locally" if row["same_sign"] else
               "DOES NOT survive locally -- may be a region A/B covariate-distribution artefact")
    print(f"{pair:24s} whole={row['ShapDiDDelta']:+.4f}  local={row['ShapDiDDelta_local']:+.4f}  "
          f"-> {verdict}")

## Per-feature view — which features moved, baseline vs each axis

Illustrative only (Steps 1-3 above are the actual estimates). One figure per (version, split,
axis) that has SHAP -- baseline vs that axis, top movers by |delta mean|phi||. Real name and
alias twin, same convention as the rest of this notebook family.

In [ ]:
# §5 — per-feature top-mover figures, baseline vs each axis, every Step 1 population
TOPK_FEATURES = 10


def _delta_fig(delta: pd.Series, label_map: pd.Series, version: str, split: str, tag: str,
               aliased: bool) -> None:
    top = delta.reindex(delta.abs().sort_values(ascending=False).index[:TOPK_FEATURES])[::-1]
    top_labels = list(label_map.reindex(top.index))
    fig, ax = plt.subplots(figsize=(6.5, 0.4 * len(top) + 1.2))
    colours = [figstyle.SERIES[0] if v >= 0 else figstyle.SERIES[1] for v in top.values]
    bars = ax.barh(np.arange(len(top)), top.values, color=colours)
    ax.set_yticks(np.arange(len(top)))
    ax.set_yticklabels(top_labels, fontsize=8)
    ax.axvline(0, color=figstyle.INK, lw=0.8)
    ax.set_xlabel("mean|phi| baseline - mean|phi| corrected")
    ax.set_title(f"{version}/{split} \u00b7 baseline -> {tag} \u00b7 top movers")
    ax.bar_label(bars, fmt="%.3f", fontsize=6, padding=2)
    fig.tight_layout()
    prefix = "alias_" if aliased else ""
    figstyle.save_fig(fig, f"{prefix}{version}_{split}_04_03_baseline_vs_{tag}_top_movers")
    plt.show()


for version, split, restrict in STEP1_POPULATIONS:
    base_table = load_region_table(version, split, "baseline")
    if restrict:
        base_table = restrict_to_corrector_targets(base_table, version, split)
    mabs_base = concentration.mean_abs(base_table, id_col=ID_COL)

    for tag in step1_effects[(version, split)]:
        if tag == "baseline":
            continue
        tag_table = load_region_table(version, split, tag)
        if restrict:
            tag_table = restrict_to_corrector_targets(tag_table, version, split)
        common = set(base_table[ID_COL]) & set(tag_table[ID_COL])
        mabs_b = concentration.mean_abs(base_table[base_table[ID_COL].isin(common)], id_col=ID_COL)
        mabs_t = concentration.mean_abs(tag_table[tag_table[ID_COL].isin(common)], id_col=ID_COL)
        feats = sorted(set(mabs_b.index) & set(mabs_t.index))
        delta = (mabs_b.reindex(feats) - mabs_t.reindex(feats)).rename("delta")

        real_labels = pd.Series(list(delta.index), index=delta.index)
        alias_labels = pd.Series(feature_alias.to_alias(version, list(delta.index)), index=delta.index)
        _delta_fig(delta, real_labels, version, split, tag, aliased=False)
        _delta_fig(delta, alias_labels, version, split, tag, aliased=True)

        table = delta.sort_values(ascending=False).to_frame("delta_mean_abs_phi")
        figstyle.save_table(table, f"{version}_{split}_04_03_baseline_vs_{tag}_feature_delta")
        alias_table = table.set_axis(feature_alias.to_alias(version, list(table.index)), axis=0).rename_axis("feature")
        figstyle.save_table(alias_table, f"alias_{version}_{split}_04_03_baseline_vs_{tag}_feature_delta")

## Caveats

- **Positivity is dead at τ.** Mitigated v3's region B (score > τ) has ZERO garage-verified rows
  by construction (`project_ips_positivity_dead`) -- Step 2/3's v3 side rests on extrapolation
  (`g(x)` transport, never a verified label). State this every time these numbers are quoted.
- **Not IPS.** Say "reweight corrector" / "the mitigation pipeline", never "IPS-corrected" --
  `mitigator/corrector/reweight.py`'s own docstring says so explicitly.
- **v2's fallback-to-baseline is per tag, printed every time** (Step 2/3) -- a `ShapDiDDelta`
  computed with v2 baseline vs v2's own corrected model compares different things even though the
  formula looks identical; check `cf_detail[tag]["v2_tag_used"]` before quoting a number.
- **Parallel trends**: this notebook no longer computes a within-version era-based confound
  check -- that lives in `04_02`'s `within_version_confound_check` (v3 TRAIN only) and `04_04`'s
  regime-break robustness. If either flags drift, Step 2/3's numbers inherit that limitation.
- **v1 is out of scope everywhere here** -- no forced-label $U$ cell to correct, and its scrap
  rule has no scalar τ to band around (`project_v1_mobility_not_a_feature`).
- Not yet run against real data for most axes/splits -- as of 2026-09-15 only `transport` has
  `mitigated_attributions` computed anywhere, and only on `train`. See the "실행 전 준비" cell
  above §1.